# CmdStanPy Conversion Guide

This guide demonstrates how to convert CmdStanPy results into an ArviZ `DataTree` using the Eight Schools model.

We illustrate:

- Basic conversion
- Specifying dimensions and coordinates
- Adding constant data
- Processing generated quantities (posterior predictive and log likelihood)

In [1]:
import numpy as np
import arviz_base as az
from cmdstanpy import CmdStanModel

### Eight Schools Data

In [2]:
J = 8
y = np.array([28, 8, -3, 7, -1, 1, 18, 12])
sigma = np.array([15, 10, 16, 11, 9, 11, 10, 18])

data = {"J": J, "y": y, "sigma": sigma}

## Stan Model

In [3]:
stan_code = """
data {
  int<lower=0> J;
  array[J] real y;
  array[J] real<lower=0> sigma;
}

parameters {
  real mu;
  real<lower=0> tau;
  array[J] real theta_tilde;
}

transformed parameters {
  array[J] real theta;
  for (j in 1:J)
    theta[j] = mu + tau * theta_tilde[j];
}

model {
  mu ~ normal(0, 5);
  tau ~ exponential(1);
  theta_tilde ~ normal(0, 1);
  y ~ normal(theta, sigma);
}

generated quantities {
  array[J] real log_lik;
  array[J] real y_rep;

  for (j in 1:J) {
    log_lik[j] = normal_lpdf(y[j] | theta[j], sigma[j]);
    y_rep[j] = normal_rng(theta[j], sigma[j]);
  }
}
"""

In [4]:
with open("eight_schools.stan", "w") as f:
    f.write(stan_code)

In [5]:
model = CmdStanModel(stan_file="eight_schools.stan")

12:52:51 - cmdstanpy - INFO - compiling stan file /Users/shivanipatel/arviz-base/docs/source/how_to/eight_schools.stan to exe file /Users/shivanipatel/arviz-base/docs/source/how_to/eight_schools
12:52:57 - cmdstanpy - INFO - compiled model executable: /Users/shivanipatel/arviz-base/docs/source/how_to/eight_schools


In [6]:
fit = model.sample(data=data, chains=2, iter_sampling=500)

12:52:57 - cmdstanpy - INFO - CmdStan start processing


chain 1:   0%|          | 0/1500 [00:00<?, ?it/s, (Warmup)]

chain 2:   0%|          | 0/1500 [00:00<?, ?it/s, (Warmup)]

12:52:57 - cmdstanpy - INFO - CmdStan done processing.


## Basic Conversion

In [7]:
idata_basic = az.from_cmdstanpy(posterior=fit)
idata_basic

<xarray.DataTree>
Group: /
├── Group: /posterior
│       Dimensions:            (chain: 2, draw: 500, theta_tilde_dim_0: 8,
│                               theta_dim_0: 8, log_lik_dim_0: 8, y_rep_dim_0: 8)
│       Coordinates:
│         * chain              (chain) int64 16B 0 1
│         * draw               (draw) int64 4kB 0 1 2 3 4 5 ... 494 495 496 497 498 499
│         * theta_tilde_dim_0  (theta_tilde_dim_0) int64 64B 0 1 2 3 4 5 6 7
│         * theta_dim_0        (theta_dim_0) int64 64B 0 1 2 3 4 5 6 7
│         * log_lik_dim_0      (log_lik_dim_0) int64 64B 0 1 2 3 4 5 6 7
│         * y_rep_dim_0        (y_rep_dim_0) int64 64B 0 1 2 3 4 5 6 7
│       Data variables:
│           mu                 (chain, draw) float64 8kB 8.36 0.05183 ... 8.138 2.571
│           tau                (chain, draw) float64 8kB 0.3623 1.29 ... 0.1558 1.113
│           theta_tilde        (chain, draw, theta_tilde_dim_0) float64 64kB -0.243 ....
│           theta              (chain, draw, theta_dim_0) float64 64kB 8.272 ... 1.275
│           log_lik            (chain, draw, log_lik_dim_0) float64 64kB -4.492 ... -...
│           y_rep              (chain, draw, y_rep_dim_0) float64 64kB 33.35 ... -2.368
│       Attributes:
│           created_at:                 2026-02-28T07:22:57.730170+00:00
│           creation_library:           ArviZ
│           creation_library_version:   0.9.0.dev0
│           creation_library_language:  Python
│           inference_library:          cmdstanpy
│           inference_library_version:  1.3.0
└── Group: /sample_stats
        Dimensions:          (chain: 2, draw: 500)
        Coordinates:
          * chain            (chain) int64 16B 0 1
          * draw             (draw) int64 4kB 0 1 2 3 4 5 6 ... 494 495 496 497 498 499
        Data variables:
            lp               (chain, draw) float64 8kB -8.068 -9.264 ... -9.349 -8.766
            acceptance_rate  (chain, draw) float64 8kB 0.9549 0.9691 ... 0.9725 0.826
            step_size        (chain, draw) float64 8kB 0.5748 0.5748 ... 0.5639 0.5639
            tree_depth       (chain, draw) int64 8kB 3 3 3 3 3 3 3 3 ... 3 3 2 3 3 3 3 3
            n_steps          (chain, draw) int64 8kB 7 7 7 7 7 7 7 7 ... 7 7 7 7 7 7 7 7
            diverging        (chain, draw) bool 1kB False False False ... False False
            energy           (chain, draw) float64 8kB 11.49 10.42 19.01 ... 15.82 17.77
        Attributes:
            created_at:                 2026-02-28T07:22:57.732109+00:00
            creation_library:           ArviZ
            creation_library_version:   0.9.0.dev0
            creation_library_language:  Python
            inference_library:          cmdstanpy
            inference_library_version:  1.3.0

### Inspecting the Resulting Groups

The converted object contains posterior draws and sampler statistics.

In [8]:
idata_basic.posterior

<xarray.DataTree 'posterior'>
Group: /posterior
    Dimensions:            (chain: 2, draw: 500, theta_tilde_dim_0: 8,
                            theta_dim_0: 8, log_lik_dim_0: 8, y_rep_dim_0: 8)
    Coordinates:
      * chain              (chain) int64 16B 0 1
      * draw               (draw) int64 4kB 0 1 2 3 4 5 ... 494 495 496 497 498 499
      * theta_tilde_dim_0  (theta_tilde_dim_0) int64 64B 0 1 2 3 4 5 6 7
      * theta_dim_0        (theta_dim_0) int64 64B 0 1 2 3 4 5 6 7
      * log_lik_dim_0      (log_lik_dim_0) int64 64B 0 1 2 3 4 5 6 7
      * y_rep_dim_0        (y_rep_dim_0) int64 64B 0 1 2 3 4 5 6 7
    Data variables:
        mu                 (chain, draw) float64 8kB 8.36 0.05183 ... 8.138 2.571
        tau                (chain, draw) float64 8kB 0.3623 1.29 ... 0.1558 1.113
        theta_tilde        (chain, draw, theta_tilde_dim_0) float64 64kB -0.243 ....
        theta              (chain, draw, theta_dim_0) float64 64kB 8.272 ... 1.275
        log_lik            (chain, draw, log_lik_dim_0) float64 64kB -4.492 ... -...
        y_rep              (chain, draw, y_rep_dim_0) float64 64kB 33.35 ... -2.368
    Attributes:
        created_at:                 2026-02-28T07:22:57.730170+00:00
        creation_library:           ArviZ
        creation_library_version:   0.9.0.dev0
        creation_library_language:  Python
        inference_library:          cmdstanpy
        inference_library_version:  1.3.0

In [9]:
idata_basic.sample_stats

<xarray.DataTree 'sample_stats'>
Group: /sample_stats
    Dimensions:          (chain: 2, draw: 500)
    Coordinates:
      * chain            (chain) int64 16B 0 1
      * draw             (draw) int64 4kB 0 1 2 3 4 5 6 ... 494 495 496 497 498 499
    Data variables:
        lp               (chain, draw) float64 8kB -8.068 -9.264 ... -9.349 -8.766
        acceptance_rate  (chain, draw) float64 8kB 0.9549 0.9691 ... 0.9725 0.826
        step_size        (chain, draw) float64 8kB 0.5748 0.5748 ... 0.5639 0.5639
        tree_depth       (chain, draw) int64 8kB 3 3 3 3 3 3 3 3 ... 3 3 2 3 3 3 3 3
        n_steps          (chain, draw) int64 8kB 7 7 7 7 7 7 7 7 ... 7 7 7 7 7 7 7 7
        diverging        (chain, draw) bool 1kB False False False ... False False
        energy           (chain, draw) float64 8kB 11.49 10.42 19.01 ... 15.82 17.77
    Attributes:
        created_at:                 2026-02-28T07:22:57.732109+00:00
        creation_library:           ArviZ
        creation_library_version:   0.9.0.dev0
        creation_library_language:  Python
        inference_library:          cmdstanpy
        inference_library_version:  1.3.0

## Specifying Dimensions and Coordinates

We can provide coordinate labels and dimension names to make the posterior output easier to interpret.

In [10]:
schools = [
    "Choate", "Deerfield", "Phillips Andover",
    "Phillips Exeter", "Hotchkiss",
    "Lawrenceville", "St. Paul's", "Mt. Hermon"
]

coords = {"school": schools}
dims = {"theta": ["school"]}

idata_dims = az.from_cmdstanpy(
    posterior=fit,
    coords=coords,
    dims=dims
)

idata_dims.posterior

<xarray.DataTree 'posterior'>
Group: /posterior
    Dimensions:            (chain: 2, draw: 500, theta_tilde_dim_0: 8, school: 8,
                            log_lik_dim_0: 8, y_rep_dim_0: 8)
    Coordinates:
      * chain              (chain) int64 16B 0 1
      * draw               (draw) int64 4kB 0 1 2 3 4 5 ... 494 495 496 497 498 499
      * theta_tilde_dim_0  (theta_tilde_dim_0) int64 64B 0 1 2 3 4 5 6 7
      * school             (school) <U16 512B 'Choate' 'Deerfield' ... 'Mt. Hermon'
      * log_lik_dim_0      (log_lik_dim_0) int64 64B 0 1 2 3 4 5 6 7
      * y_rep_dim_0        (y_rep_dim_0) int64 64B 0 1 2 3 4 5 6 7
    Data variables:
        mu                 (chain, draw) float64 8kB 8.36 0.05183 ... 8.138 2.571
        tau                (chain, draw) float64 8kB 0.3623 1.29 ... 0.1558 1.113
        theta_tilde        (chain, draw, theta_tilde_dim_0) float64 64kB -0.243 ....
        theta              (chain, draw, school) float64 64kB 8.272 8.202 ... 1.275
        log_lik            (chain, draw, log_lik_dim_0) float64 64kB -4.492 ... -...
        y_rep              (chain, draw, y_rep_dim_0) float64 64kB 33.35 ... -2.368
    Attributes:
        created_at:                 2026-02-28T07:22:57.786306+00:00
        creation_library:           ArviZ
        creation_library_version:   0.9.0.dev0
        creation_library_language:  Python
        inference_library:          cmdstanpy
        inference_library_version:  1.3.0

## Adding Constant Data

We can include observed data as constant_data in the resulting DataTree.

In [11]:
idata_const = az.from_cmdstanpy(
    posterior=fit,
    constant_data={"y": data["y"]}
)

idata_const.constant_data

<xarray.DataTree 'constant_data'>
Group: /constant_data
    Dimensions:  (y_dim_0: 8)
    Coordinates:
      * y_dim_0  (y_dim_0) int64 64B 0 1 2 3 4 5 6 7
    Data variables:
        y        (y_dim_0) int64 64B 28 8 -3 7 -1 1 18 12
    Attributes:
        created_at:                 2026-02-28T07:22:57.804250+00:00
        creation_library:           ArviZ
        creation_library_version:   0.9.0.dev0
        creation_library_language:  Python
        inference_library:          cmdstanpy
        inference_library_version:  1.3.0

## Processing Generated Quantities (Posterior Predictive and Log Likelihood)

In [12]:
idata_full = az.from_cmdstanpy(
    posterior=fit,
    posterior_predictive="y_rep",
    log_likelihood="log_lik",
    observed_data={"y": data["y"]},
)

idata_full

<xarray.DataTree>
Group: /
├── Group: /observed_data
│       Dimensions:  (y_dim_0: 8)
│       Coordinates:
│         * y_dim_0  (y_dim_0) int64 64B 0 1 2 3 4 5 6 7
│       Data variables:
│           y        (y_dim_0) int64 64B 28 8 -3 7 -1 1 18 12
│       Attributes:
│           created_at:                 2026-02-28T07:22:57.817668+00:00
│           creation_library:           ArviZ
│           creation_library_version:   0.9.0.dev0
│           creation_library_language:  Python
│           inference_library:          cmdstanpy
│           inference_library_version:  1.3.0
├── Group: /posterior
│       Dimensions:            (chain: 2, draw: 500, theta_tilde_dim_0: 8,
│                               theta_dim_0: 8)
│       Coordinates:
│         * chain              (chain) int64 16B 0 1
│         * draw               (draw) int64 4kB 0 1 2 3 4 5 ... 494 495 496 497 498 499
│         * theta_tilde_dim_0  (theta_tilde_dim_0) int64 64B 0 1 2 3 4 5 6 7
│         * theta_dim_0        (theta_dim_0) int64 64B 0 1 2 3 4 5 6 7
│       Data variables:
│           mu                 (chain, draw) float64 8kB 8.36 0.05183 ... 8.138 2.571
│           tau                (chain, draw) float64 8kB 0.3623 1.29 ... 0.1558 1.113
│           theta_tilde        (chain, draw, theta_tilde_dim_0) float64 64kB -0.243 ....
│           theta              (chain, draw, theta_dim_0) float64 64kB 8.272 ... 1.275
│       Attributes:
│           created_at:                 2026-02-28T07:22:57.819047+00:00
│           creation_library:           ArviZ
│           creation_library_version:   0.9.0.dev0
│           creation_library_language:  Python
│           inference_library:          cmdstanpy
│           inference_library_version:  1.3.0
├── Group: /sample_stats
│       Dimensions:          (chain: 2, draw: 500)
│       Coordinates:
│         * chain            (chain) int64 16B 0 1
│         * draw             (draw) int64 4kB 0 1 2 3 4 5 6 ... 494 495 496 497 498 499
│       Data variables:
│           lp               (chain, draw) float64 8kB -8.068 -9.264 ... -9.349 -8.766
│           acceptance_rate  (chain, draw) float64 8kB 0.9549 0.9691 ... 0.9725 0.826
│           step_size        (chain, draw) float64 8kB 0.5748 0.5748 ... 0.5639 0.5639
│           tree_depth       (chain, draw) int64 8kB 3 3 3 3 3 3 3 3 ... 3 3 2 3 3 3 3 3
│           n_steps          (chain, draw) int64 8kB 7 7 7 7 7 7 7 7 ... 7 7 7 7 7 7 7 7
│           diverging        (chain, draw) bool 1kB False False False ... False False
│           energy           (chain, draw) float64 8kB 11.49 10.42 19.01 ... 15.82 17.77
│       Attributes:
│           created_at:                 2026-02-28T07:22:57.820798+00:00
│           creation_library:           ArviZ
│           creation_library_version:   0.9.0.dev0
│           creation_library_language:  Python
│           inference_library:          cmdstanpy
│           inference_library_version:  1.3.0
├── Group: /posterior_predictive
│       Dimensions:      (chain: 2, draw: 500, y_rep_dim_0: 8)
│       Coordinates:
│         * chain        (chain) int64 16B 0 1
│         * draw         (draw) int64 4kB 0 1 2 3 4 5 6 ... 493 494 495 496 497 498 499
│         * y_rep_dim_0  (y_rep_dim_0) int64 64B 0 1 2 3 4 5 6 7
│       Data variables:
│           y_rep        (chain, draw, y_rep_dim_0) float64 64kB 33.35 4.224 ... -2.368
│       Attributes:
│           created_at:                 2026-02-28T07:22:57.821819+00:00
│           creation_library:           ArviZ
│           creation_library_version:   0.9.0.dev0
│           creation_library_language:  Python
│           inference_library:          cmdstanpy
│           inference_library_version:  1.3.0
└── Group: /log_likelihood
        Dimensions:        (chain: 2, draw: 500, log_lik_dim_0: 8)
        Coordinates:
          * chain          (chain) int64 16B 0 1
          * draw           (draw) int64 4kB 0 1 2 3 4 5 6 ... 494 495 496 497 498 499
          * log_lik_dim_0  (log_lik_di